# ROAD-SHIELD: train the scene detectors on a Colab GPU

Trains the three YOLO detectors that `models/road_scene_perception.py` serves:

| Model | Classes | Data |
|---|---|---|
| `road_damage` | longitudinal, transverse, alligator cracks; potholes | RDD2022, Indian subset (~7,700 images) |
| `crosswalk` | zebra crossings, lane guide arrows | CDSet-3434, re-split without video leakage |
| `license_plate` | licence plates (for blurring only) | Vehicle Registration Plates, 8,823 images |

It runs the **same code as the repository** (`scripts/fetch_rdd2022.py`,
`scripts/prepare_crosswalk_dataset.py`, `training/train_detector.py`), so the
model cards, per-class thresholds and test metrics are produced exactly as they
would be locally, just with a GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Expected time on a free T4: road damage ~60–90 min, crosswalk ~25–35 min, plates ~30–40 min.

Runs are saved to Google Drive after every epoch. If Colab disconnects, re-run
the cells from the top with `RESUME = True` in the training cell; the data
cells skip anything already downloaded.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

Pick **one** source:

* `"github"`: clones the branch. Push it first from your machine:
  `git push -u origin feature/road-scene-perception`
* `"upload"`: upload `road_shield_code.zip` (made with
  `git archive -o road_shield_code.zip HEAD -- . ":(exclude)datasets"`) when prompted.

In [ ]:
SOURCE = "github"            # "github" or "upload"
REPO_URL = "https://github.com/udbhav968-creator/SIH_PROJECT.git"
BRANCH = "feature/road-scene-perception"

import os, shutil, subprocess
os.chdir("/content")
if os.path.exists("SIH_PROJECT"):
    print("code already present")
elif SOURCE == "github":
    # Blobless + sparse: skips the ~300 MB bundled datasets/ folder, which training does not need.
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
                    "-b", BRANCH, REPO_URL, "SIH_PROJECT"], check=True)
    subprocess.run("cd SIH_PROJECT && git sparse-checkout set --no-cone '/*' '!/datasets/' && git checkout",
                   shell=True, check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    archive = next(iter(uploaded))
    shutil.unpack_archive(archive, "SIH_PROJECT")
os.chdir("/content/SIH_PROJECT")
os.makedirs("datasets", exist_ok=True)
print(os.getcwd(), sorted(os.listdir("."))[:12])

## 2. Install

Colab already ships PyTorch with CUDA; this adds Ultralytics and the export/serving tools.

In [ ]:
!pip install -q -r requirements-train.txt pyyaml onnxruntime
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 3. Keep runs on Google Drive

Checkpoints are written every epoch, so a disconnect costs at most one epoch.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/road_shield_runs"
os.makedirs(RUNS, exist_ok=True)
print("runs ->", RUNS)

## 4. Road damage: data

Downloads the Indian subset of RDD2022 (CC BY-SA 4.0). Resumable. If
Hugging Face rate-limits you (HTTP 429), re-run the cell, or add an `HF_TOKEN`
under Colab's 🔑 Secrets and uncomment the first line.

In [ ]:
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!python -m scripts.fetch_rdd2022 --out datasets/rdd2022_india --countries India --train-background-fraction 0.33 --workers 8

## 5. Road damage: train

Defaults below are sized for a T4. `patience` in the config stops early if
validation mAP stops improving. After training, the script tunes thresholds on
the validation split, scores the test split, exports ONNX and writes the model card.

In [ ]:
RESUME = False   # set True after a disconnect
EPOCHS, IMGSZ, BATCH = 100, 640, 32

resume_flag = "--resume" if RESUME else ""
!python -m training.train_detector configs/detectors/road_damage.yaml     --device 0 --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --runs-dir {RUNS} {resume_flag}

## 6. Zebra crossings: data

CDSet-3434 (Apache-2.0), ~1.1 GB. The preparation script re-splits it by time
block per video, because the official test frames sit 1–2 frames from
training frames and would inflate every score.

In [ ]:
if not os.path.isdir("datasets/CDSet"):
    !wget -q --show-progress -O datasets/CDSet.zip https://huggingface.co/datasets/zzd0225/crosswalk-detection-dataset/resolve/main/CDSet.zip
    !unzip -q datasets/CDSet.zip -d datasets/ && rm datasets/CDSet.zip
!python -m scripts.prepare_crosswalk_dataset --src datasets/CDSet --out datasets/crosswalk

## 7. Zebra crossings: train

In [ ]:
RESUME = False   # set True after a disconnect
EPOCHS, IMGSZ, BATCH = 60, 512, 32

resume_flag = "--resume" if RESUME else ""
!python -m training.train_detector configs/detectors/crosswalk.yaml     --device 0 --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --runs-dir {RUNS} {resume_flag}

## 7b. Licence plates (privacy redaction)

Trains the detector used only to **blur** plates before an image leaves a bus
(DPDP Act 2023); plates are never read. 8,823 images, CC BY 4.0. The 3,904
synthetic rendered plates are kept in training only, so the test score is on
real photographs. ~30–40 min on a T4.

In [ ]:
!python -m scripts.fetch_license_plates --out datasets/license_plates

In [ ]:
RESUME = False   # set True after a disconnect
EPOCHS, IMGSZ, BATCH = 60, 640, 32

resume_flag = "--resume" if RESUME else ""
!python -m training.train_detector configs/detectors/license_plate.yaml --device 0 --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --runs-dir {RUNS} {resume_flag}

## 8. Head-to-head: learned vs geometric crossing detector

Scores the exported ONNX model (what the server loads) and the old
geometric bar finder on the same held-out frames.

In [ ]:
!python -m scripts.benchmark_crosswalk

## 9. Results

In [ ]:
import json, glob
for card in sorted(glob.glob("checkpoints/detectors/*.json")):
    d = json.load(open(card))
    if "metrics" not in d:
        continue
    t, img = d["metrics"]["test"], d["metrics"]["test_image_level"]
    print(f"
== {d['name']}  ({d['hardware']}, {d['config']['epochs']} epochs @ {d['input_size']}px)")
    print(f"test mAP50 {t['mAP50']:.3f} | mAP50-95 {t['mAP50_95']:.3f} | P {t['precision']:.3f} | R {t['recall']:.3f}")
    print("clean-road false-alarm rate:", img["background_false_alarm_rate"])
    for cls, m in t["per_class"].items():
        print(f"  {cls:20s} mAP50 {m['mAP50']:.3f}  threshold {d['serving_thresholds'][cls]}")

## 10. Download the trained models

Copies `checkpoints/detectors/` to Drive and downloads a zip.

**To install on your machine:** unzip into `SIH_PROJECT/checkpoints/detectors/`
(so you have `road_damage.onnx`, `road_damage.json`, `crosswalk.onnx`,
`crosswalk.json`, ...) and restart `python -m api.server`. The `/detect` page and
`/api/v1/perception/analyze` pick them up on start, and
`python -m unittest discover -s tests` runs the trained-detector integration
tests instead of skipping them.

In [ ]:
shutil.copytree("checkpoints/detectors", f"{RUNS}/detectors_export", dirs_exist_ok=True)
shutil.make_archive("/content/road_shield_detectors", "zip", "checkpoints/detectors")
from google.colab import files
files.download("/content/road_shield_detectors.zip")